SECTION 1: SETUP

In [1]:
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
import dask

import pandas as pd
import numpy as np
import pyarrow as pa
import os
import pickle
from datetime import datetime
import gc
import shutil
import time

dask.config.set({
    'dataframe.shuffle.method': 'tasks',
    'distributed.worker.memory.target': 0.6,
    'distributed.worker.memory.spill': 0.7,
    'distributed.worker.memory.pause': 0.8,
    'distributed.worker.memory.terminate': 0.95,
})

print(f"Dask version: {dask.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"PyArrow version: {pa.__version__}")
print("\nSetup complete!")

Dask version: 2026.1.2
Pandas version: 2.2.2
PyArrow version: 23.0.0

Setup complete!


SECTION 2: CONFIGURATION

In [2]:
DATA_DIR = r"C:\MATH699P\Data"


PROCESSED_DIR = os.path.join(DATA_DIR, 'processed_data')
FEATURES_DIR = os.path.join(PROCESSED_DIR, 'features_parquet', 'features_engineered')
MODEL_DATA_DIR = os.path.join(PROCESSED_DIR, 'model_ready_dask')

os.makedirs(MODEL_DATA_DIR, exist_ok=True)

CONFIG = {
    'target': 'OZONE',
    'train_end': '2019-12-31',
    'val_end': '2022-12-31',
    'sequence_length': 24,
    'forecast_horizon': 1,
}

TARGET_COL = CONFIG['target']

print("Configuration:")
print("-" * 60)
for k, v in CONFIG.items():
    print(f"{k}: {v}")
print(f"\nInput:  {FEATURES_DIR}")
print(f"Output: {MODEL_DATA_DIR}")

Configuration:
------------------------------------------------------------
target: OZONE
train_end: 2019-12-31
val_end: 2022-12-31
sequence_length: 24
forecast_horizon: 1

Input:  C:\MATH699P\Data\processed_data\features_parquet\features_engineered
Output: C:\MATH699P\Data\processed_data\model_ready_dask


SECTION 3: LOAD DATA LAZILY AND REPARTITION

In [3]:
print("LOADING DATA (LAZY)")
print("=" * 60)

# Load as Dask DataFrame - LAZY
ddf = dd.read_parquet(FEATURES_DIR)

print(f"Columns: {len(ddf.columns)}")
print(f"Original partitions: {ddf.npartitions}")

TARGET_PARTITION_SIZE = "150MB"
ddf = ddf.repartition(partition_size=TARGET_PARTITION_SIZE)

print(f"Repartitioned to: {ddf.npartitions} partitions (~{TARGET_PARTITION_SIZE} each)")
print(f"\nData types (first 10):")
print(ddf.dtypes.head(10))

LOADING DATA (LAZY)
Columns: 75
Original partitions: 43
Repartitioned to: 62 partitions (~150MB each)

Data types (first 10):
SITE_ID      string[pyarrow]
DATE_TIME     datetime64[ns]
OZONE                float32
QA_CODE                int32
year                   int16
month                   int8
day                     int8
hour                    int8
dayofweek               int8
dayofyear              int16
dtype: object


SECTION 4: DEFINE FEATURE COLUMNS

In [4]:
print("DEFINING FEATURE COLUMNS")
print("=" * 60)

ID_COLS = ['SITE_ID', 'DATE_TIME']

EXCLUDE_COLS = ID_COLS + [TARGET_COL, 'QA_CODE', 'hour_category', 'season',
                          'LAND_USE', 'TERRAIN', 'STATE']

numeric_dtypes = ['int8', 'int16', 'int32', 'int64', 'float16', 'float32', 'float64']
FEATURE_COLS = [col for col in ddf.columns 
                if str(ddf[col].dtype) in numeric_dtypes
                and col not in EXCLUDE_COLS]

print(f"\nFeature columns: {len(FEATURE_COLS)}")
print(f"\nFeature list:")
for i, col in enumerate(FEATURE_COLS):
    print(f"{i+1:3d}. {col}")

DEFINING FEATURE COLUMNS

Feature columns: 67

Feature list:
  1. year
  2. month
  3. day
  4. hour
  5. dayofweek
  6. dayofyear
  7. week
  8. quarter
  9. is_weekend
 10. is_rush_hour
 11. hour_sin
 12. hour_cos
 13. month_sin
 14. month_cos
 15. dayofweek_sin
 16. dayofweek_cos
 17. dayofyear_sin
 18. dayofyear_cos
 19. OZONE_lag_1
 20. OZONE_lag_2
 21. OZONE_lag_3
 22. OZONE_lag_6
 23. OZONE_lag_12
 24. OZONE_lag_24
 25. OZONE_lag_48
 26. OZONE_lag_168
 27. OZONE_rolling_mean_3
 28. OZONE_rolling_std_3
 29. OZONE_rolling_min_3
 30. OZONE_rolling_max_3
 31. OZONE_rolling_range_3
 32. OZONE_rolling_mean_6
 33. OZONE_rolling_std_6
 34. OZONE_rolling_min_6
 35. OZONE_rolling_max_6
 36. OZONE_rolling_range_6
 37. OZONE_rolling_mean_12
 38. OZONE_rolling_std_12
 39. OZONE_rolling_min_12
 40. OZONE_rolling_max_12
 41. OZONE_rolling_range_12
 42. OZONE_rolling_mean_24
 43. OZONE_rolling_std_24
 44. OZONE_rolling_min_24
 45. OZONE_rolling_max_24
 46. OZONE_rolling_range_24
 47. OZONE_roll

SECTION 5: COMPUTE STATISTICS FOR NORMALIZATION

In [5]:
print("FILTERING TO TRAINING DATA")
print("=" * 60)

train_end = pd.to_datetime(CONFIG['train_end'])

ddf_train = ddf[ddf['DATE_TIME'] <= train_end]

ddf_train = ddf_train.dropna(subset=[TARGET_COL])

print(f"Filtered to training period (up to {CONFIG['train_end']})")
print(f"Removed rows with missing {TARGET_COL}")
print(f"(Still lazy - no data loaded yet)")

FILTERING TO TRAINING DATA
Filtered to training period (up to 2019-12-31)
Removed rows with missing OZONE
(Still lazy - no data loaded yet)


In [6]:
print("\nCOMPUTING FEATURE STATISTICS")
print("=" * 60)

print("\nComputing means...")
with ProgressBar():
    feature_means = ddf_train[FEATURE_COLS].mean().compute()
print(f"Mean range: [{feature_means.min():.4f}, {feature_means.max():.4f}]")

print("\nComputing standard deviations...")
with ProgressBar():
    feature_stds = ddf_train[FEATURE_COLS].std().compute()
feature_stds = feature_stds.replace(0, 1).fillna(1)
print(f"Std range: [{feature_stds.min():.4f}, {feature_stds.max():.4f}]")

print("\nComputing medians from 5% sample...")
with ProgressBar():
    sample = ddf_train[FEATURE_COLS].sample(frac=0.05, random_state=42).compute()
feature_medians = sample.median()
print(f"Computed from {len(sample):,} sampled rows")

del sample
gc.collect()

feature_means_dict = feature_means.to_dict()
feature_stds_dict = feature_stds.to_dict()
feature_medians_dict = feature_medians.to_dict()

print(f"\nStatistics computed for {len(FEATURE_COLS)} features")


COMPUTING FEATURE STATISTICS

Computing means...
[########################################] | 100% Completed | 11.34 s
[########################################] | 100% Completed | 23.72 s
Mean range: [-91.3783, 2004.9736]

Computing standard deviations...
[########################################] | 100% Completed | 47.24 s
Std range: [0.4307, 774.2603]

Computing medians from 5% sample...
[########################################] | 100% Completed | 16.77 s
Computed from 909,687 sampled rows

Statistics computed for 67 features


SECTION 6: CREATE NORMALIZATION PIPELINE WITH CORRECT META

In [7]:
def normalize_partition(df, feature_cols, means, stds, medians):
    """
    Normalize a single partition (Pandas DataFrame).
    
    This function is applied to each Dask partition independently.
    
    Steps:
    1. Fill missing values with median
    2. Normalize: (value - mean) / std
    3. Convert to float32 for memory efficiency
    """
    df = df.copy()
    
    for col in feature_cols:
        if col in df.columns:
            median_val = medians.get(col, 0)
            df[col] = df[col].fillna(median_val)
            
            mean_val = means.get(col, 0)
            std_val = stds.get(col, 1)
            df[col] = ((df[col] - mean_val) / std_val).astype('float32')
    
    return df


In [8]:
print("CREATING NORMALIZATION PIPELINE")
print("=" * 60)

ID_COLS = ['SITE_ID', 'DATE_TIME']

cols_to_keep = ID_COLS + [TARGET_COL] + FEATURE_COLS
cols_available = [c for c in cols_to_keep if c in ddf.columns]

print(f"Columns to keep: {len(cols_to_keep)}")
print(f"Columns available: {len(cols_available)}")
print(f"SITE_ID included: {'SITE_ID' in cols_available}")
print(f"DATE_TIME included: {'DATE_TIME' in cols_available}")

if 'SITE_ID' not in cols_available:
    print("   WARNING: SITE_ID not found! Check column names:")
    print(f"{[c for c in ddf.columns if 'SITE' in c.upper() or 'ID' in c.upper()]}")

ddf_selected = ddf[cols_available]

ddf_clean = ddf_selected.dropna(subset=[TARGET_COL])

meta_fixed = ddf_clean._meta.copy()
for col in FEATURE_COLS:
    if col in meta_fixed.columns:
        meta_fixed[col] = meta_fixed[col].astype('float32')

print(f"Created output schema (meta) with float32 features")

ddf_normalized = ddf_clean.map_partitions(
    normalize_partition,
    feature_cols=FEATURE_COLS,
    means=feature_means_dict,
    stds=feature_stds_dict,
    medians=feature_medians_dict,
    meta=meta_fixed 
)

print(f"\nNormalization pipeline created (still lazy)")

CREATING NORMALIZATION PIPELINE
Columns to keep: 70
Columns available: 70
SITE_ID included: True
DATE_TIME included: True
Created output schema (meta) with float32 features

Normalization pipeline created (still lazy)


In [9]:
# PyArrow schema to avoid inference OOM
save_schema = pa.Schema.from_pandas(meta_fixed)
print(f"Schema has {len(save_schema)} fields")
print(f"First 5 fields: {[save_schema.field(i).name for i in range(min(5, len(save_schema)))]}")

Schema has 70 fields
First 5 fields: ['SITE_ID', 'DATE_TIME', 'OZONE', 'year', 'month']


SECTION 7: SAVE TRAIN / VALIDATION / TEST SPLITS

In [10]:
def safe_remove_dir(path, max_retries=3):
    """Safely remove directory with retries for Windows/OneDrive."""
    for attempt in range(max_retries):
        try:
            if os.path.exists(path):
                shutil.rmtree(path)
            return True
        except PermissionError:
            print(f"Attempt {attempt + 1}: Permission denied, waiting 2s...")
            time.sleep(2)
            try:
                for root, dirs, files in os.walk(path):
                    for f in files:
                        os.chmod(os.path.join(root, f), 0o777)
            except:
                pass
    return False

def count_parquet_rows(path):
    """Count rows in a parquet directory without loading all data."""
    import pyarrow.parquet as pq
    total = 0
    for f in os.listdir(path):
        if f.endswith('.parquet'):
            pf = pq.ParquetFile(os.path.join(path, f))
            total += pf.metadata.num_rows
    return total


In [11]:
print("SAVING TRAINING DATA")
print("=" * 60)

train_end = pd.to_datetime(CONFIG['train_end'])
train_output = os.path.join(MODEL_DATA_DIR, 'train')

if not safe_remove_dir(train_output):
    train_output = os.path.join(MODEL_DATA_DIR, f'train_{int(time.time())}')
    print(f"Using alternative path: {train_output}")

ddf_train_norm = ddf_normalized[ddf_normalized['DATE_TIME'] <= train_end]

print(f"Saving to: {train_output}")

with ProgressBar():
    ddf_train_norm.to_parquet(
        train_output,
        engine='pyarrow',
        schema=save_schema,  
        compression='snappy',
        write_index=False
    )

train_count = count_parquet_rows(train_output)
print(f"\nSaved {train_count:,} training rows")

gc.collect()

SAVING TRAINING DATA
Saving to: C:\MATH699P\Data\processed_data\model_ready_dask\train
[########################################] | 100% Completed | 5.69 ss
[########################################] | 100% Completed | 32.82 s

Saved 18,193,749 training rows


1099

In [12]:
print("\nSAVING VALIDATION DATA")
print("=" * 60)

val_end = pd.to_datetime(CONFIG['val_end'])
val_output = os.path.join(MODEL_DATA_DIR, 'val')

if not safe_remove_dir(val_output):
    val_output = os.path.join(MODEL_DATA_DIR, f'val_{int(time.time())}')
    print(f"Using alternative path: {val_output}")

ddf_val_norm = ddf_normalized[
    (ddf_normalized['DATE_TIME'] > train_end) & 
    (ddf_normalized['DATE_TIME'] <= val_end)
]

print(f"Saving to: {val_output}")

with ProgressBar():
    ddf_val_norm.to_parquet(
        val_output,
        engine='pyarrow',
        schema=save_schema,
        compression='snappy',
        write_index=False
    )

val_count = count_parquet_rows(val_output)
print(f"\nSaved {val_count:,} validation rows")

gc.collect()


SAVING VALIDATION DATA
Saving to: C:\MATH699P\Data\processed_data\model_ready_dask\val
[########################################] | 100% Completed | 31.07 s

Saved 2,072,797 validation rows


1311

In [13]:
print("\nSAVING TEST DATA")
print("=" * 60)

test_output = os.path.join(MODEL_DATA_DIR, 'test')

if not safe_remove_dir(test_output):
    test_output = os.path.join(MODEL_DATA_DIR, f'test_{int(time.time())}')
    print(f"Using alternative path: {test_output}")

ddf_test_norm = ddf_normalized[ddf_normalized['DATE_TIME'] > val_end]

print(f"Saving to: {test_output}")

with ProgressBar():
    ddf_test_norm.to_parquet(
        test_output,
        engine='pyarrow',
        schema=save_schema,
        compression='snappy',
        write_index=False
    )

test_count = count_parquet_rows(test_output)
print(f"\nSaved {test_count:,} test rows")

gc.collect()


SAVING TEST DATA
Saving to: C:\MATH699P\Data\processed_data\model_ready_dask\test
[########################################] | 100% Completed | 30.07 s

Saved 1,400,484 test rows


1115

SECTION 8: SAVE PREPROCESSING OBJECTS

In [14]:
print("\nSAVING PREPROCESSING OBJECTS")
print("=" * 60)

preprocessing = {
    'feature_means': feature_means_dict,
    'feature_stds': feature_stds_dict,
    'feature_medians': feature_medians_dict,
    
    'feature_cols': FEATURE_COLS,
    'id_cols': ID_COLS,
    'target_col': TARGET_COL,
    'n_features': len(FEATURE_COLS),
    
    'config': CONFIG,
    
    'train_count': train_count,
    'val_count': val_count,
    'test_count': test_count,
    
    'train_path': train_output,
    'val_path': val_output,
    'test_path': test_output,
    
}

preprocessing_path = os.path.join(MODEL_DATA_DIR, 'preprocessing.pkl')
with open(preprocessing_path, 'wb') as f:
    pickle.dump(preprocessing, f)
print(f"Saved: preprocessing.pkl")

feature_list_path = os.path.join(MODEL_DATA_DIR, 'feature_list.txt')
with open(feature_list_path, 'w') as f:
    f.write(f"FEATURE LIST ({len(FEATURE_COLS)} features)\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Target: {TARGET_COL}\n\n")
    for i, col in enumerate(FEATURE_COLS):
        f.write(f"{i+1:4d}. {col}\n")
print(f"Saved: feature_list.txt")


SAVING PREPROCESSING OBJECTS
Saved: preprocessing.pkl
Saved: feature_list.txt


SECTION 9: VERIFICATION

In [15]:
print("\nVERIFICATION")
print("=" * 60)

# Load a sample from training data to verify normalization
ddf_verify = dd.read_parquet(train_output)
sample = ddf_verify[FEATURE_COLS].head(10000)

print(f"\nNormalization check:")
print(f"Mean of features: {sample.mean().mean():.6f}")
print(f"Std of features:  {sample.std().mean():.6f}")

print(f"\nSample dtypes:")
print(sample.dtypes.head(10))


VERIFICATION

Normalization check:
Mean of features: 0.230667
Std of features:  0.963919

Sample dtypes:
year            float32
month           float32
day             float32
hour            float32
dayofweek       float32
dayofyear       float32
week            float32
quarter         float32
is_weekend      float32
is_rush_hour    float32
dtype: object


## SECTION 10: SUMMARY

In [16]:
print("\n" + "=" * 60)
print("MODEL PREPARATION COMPLETE")
print("=" * 60)

print(f"\n1. FEATURES:")
print(f"Total features: {len(FEATURE_COLS)}")
print(f"Target: {TARGET_COL}")

print(f"\n2. DATA SPLITS:")
print(f"Train:      {train_count:>12,} rows (up to {CONFIG['train_end']})")
print(f"Validation: {val_count:>12,} rows ({CONFIG['train_end']} to {CONFIG['val_end']})")
print(f"Test:       {test_count:>12,} rows (after {CONFIG['val_end']})")
print(f"Total:      {train_count + val_count + test_count:>12,} rows")

print(f"\n3. OUTPUT FILES:")
total_size = 0
for item in os.listdir(MODEL_DATA_DIR):
    item_path = os.path.join(MODEL_DATA_DIR, item)
    if os.path.isdir(item_path):
        dir_size = sum(os.path.getsize(os.path.join(item_path, f)) 
                      for f in os.listdir(item_path) if os.path.isfile(os.path.join(item_path, f)))
        total_size += dir_size
        print(f"{item}/ ({dir_size / 1e6:.1f} MB)")
    else:
        file_size = os.path.getsize(item_path)
        total_size += file_size
        print(f"{item} ({file_size / 1e3:.1f} KB)")
print(f"Total: {total_size / 1e9:.2f} GB")

print(f"\n4. NEXT STEPS:")
print(f"1. Run 'create_sequences_memory_efficient.ipynb' to create LSTM sequences")
print(f"2. Run 'model_training.ipynb' to train models")

print(f"\n" + "=" * 60)
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)


MODEL PREPARATION COMPLETE

1. FEATURES:
Total features: 67
Target: OZONE

2. DATA SPLITS:
Train:        18,193,749 rows (up to 2019-12-31)
Validation:    2,072,797 rows (2019-12-31 to 2022-12-31)
Test:          1,400,484 rows (after 2022-12-31)
Total:        21,667,030 rows

3. OUTPUT FILES:
feature_list.txt (1.7 KB)
preprocessing.pkl (6.1 KB)
test/ (84.3 MB)
train/ (1081.1 MB)
val/ (122.1 MB)
Total: 1.29 GB

4. NEXT STEPS:
1. Run 'create_sequences_memory_efficient.ipynb' to create LSTM sequences
2. Run 'model_training.ipynb' to train models

Completed at: 2026-02-03 09:06:09
